# Basic Multi Layer Perceptron using Iris Dataset

## a. Lab requirements

The second lab assignment is to use scikit-learn to implement a neural network for the Iris dataset, which is available on the Moodle page.

1. Construct a neural network consisting of:

    • an input layer with 4 nodes

    • at least one hidden layer (with, say, 20 nodes)

    • an output layer with 3 nodes. You may add more hidden layers if you wish, but at least one is required.

2. Prepare the Iris dataset by:

    (a) normalising the input values using max-min normalisation.

    (b) creating a one-hot encoding of the target values.

3. Split the dataset into training, validation, and test sets.

4. Train the neural network on the prepared Iris dataset. Use the sum-of-squares loss
function.

5. During training, after every epoch, print:

    • the sum-of-squares loss on the training set.

    • the sum-of-squares loss on the validation set.

6. After training, print the accuracy on the test set.

## b. Imports

In [74]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import mean_squared_error, accuracy_score

## c. Load Data

In [75]:
#load data locally
cols = ["x1", "x2", "x3", "x4", "target"]
df = pd.read_csv("Iris.csv", sep=";", header=None, names=cols)
df.head()

,x1,x2,x3,x4,target
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


## d. Data Cleaning

In [76]:
#Checking the data profile
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   x1      150 non-null    float64
 1   x2      150 non-null    float64
 2   x3      150 non-null    float64
 3   x4      150 non-null    float64
 4   target  150 non-null    str    
dtypes: float64(4), str(1)
memory usage: 6.0 KB


### Key Takeaways:

1. We have no missing values in any of the columns.

2. The target column is of string type. Need conversion to integer before modeling; One hot encoding has been recommended in the requirements.



In [77]:
#check for duplicates
total_duplicates = df.duplicated().sum()
print(int(total_duplicates))

3


We have three duplicates in our dataset.

In [78]:
df[df.duplicated()]

,x1,x2,x3,x4,target
34,4.9,3.1,1.5,0.1,Iris-setosa
37,4.9,3.1,1.5,0.1,Iris-setosa
142,5.8,2.7,5.1,1.9,Iris-virginica


In [79]:
#dropt the duplicates
df.drop_duplicates(inplace=True)
df.info()

<class 'pandas.DataFrame'>
Index: 147 entries, 0 to 149
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   x1      147 non-null    float64
 1   x2      147 non-null    float64
 2   x3      147 non-null    float64
 3   x4      147 non-null    float64
 4   target  147 non-null    str    
dtypes: float64(4), str(1)
memory usage: 6.9 KB


I dropped all the duplicates keepiing the first occurances only. This reduced or dataset to 147 from 150 values.

In [80]:
#Checking the summary stats
df.describe()

,x1,x2,x3,x4
count,147.000000,147.000000,147.000000,147.000000
mean,5.856463,3.055782,3.780272,1.208844
std,0.829100,0.437009,1.759111,0.757874
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.400000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


In [81]:
df.groupby(["target"])["target"].count()

target
Iris-setosa        48
Iris-versicolor    50
Iris-virginica     49
Name: target, dtype: int64

The target column is pretty much balanced.

## e. Split the Raw Data

In [82]:
#Split to features and target
X = df[["x1", "x2", "x3", "x4"]]
y= df["target"]


#split to train & validation  and test sets (80/20)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, stratify=y, shuffle=True, random_state=42)

#split the training and validation sets (75/25)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, stratify=y_train_val, shuffle=True, random_state=42)

## f. Data Preprocessing

### Feature Scaling and Normalization

In [83]:
#Istatnitate the scaler
scaler = MinMaxScaler()

# Fit only on training data, transform all other three sets
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

### One hot Encoding

In [84]:
encoder = OneHotEncoder(sparse_output=False)

# Fit on training labels, transform all three
y_train_encoded = encoder.fit_transform(y_train.values.reshape(-1, 1))
y_val_encoded = encoder.transform(y_val.values.reshape(-1, 1))
y_test_encoded = encoder.transform(y_test.values.reshape(-1, 1))

## g. Modeling

### Building the Multilayer Neural Network

In [85]:
#instatiate
model = MLPClassifier(hidden_layer_sizes=(3,8), activation='logistic', max_iter=1, random_state=42)

In [86]:
all_classes = [0, 1, 2]
epochs = 10000
for epoch in range(epochs):
    # 1. Take exactly ONE step of gradient descent
    model.partial_fit(X_train_scaled, y_train_encoded, classes=all_classes)
    
    # 2. Get predicted probabilities for both sets
    train_preds = model.predict_proba(X_train_scaled)
    val_preds = model.predict_proba(X_val_scaled)
    
    # 3. Calculate Sum-of-Squares Error (SSE)
    # Note: mean_squared_error gives MSE. Multiplying by the number of samples gives SSE.
    train_sse = mean_squared_error(y_train_encoded, train_preds) * len(y_train_encoded)
    val_sse = mean_squared_error(y_val_encoded, val_preds) * len(y_val_encoded)
    
    # 4. Print progress
    if (epoch + 1) % 10 == 0: # Print every 10 epochs so your notebook isn't flooded
        print(f"Epoch {epoch+1:03d} -> Train SSE: {train_sse:.4f} | Val SSE: {val_sse:.4f}")

Epoch 010 -> Train SSE: 21.4251 | Val SSE: 7.3731
Epoch 020 -> Train SSE: 21.1241 | Val SSE: 7.2697
Epoch 030 -> Train SSE: 20.8579 | Val SSE: 7.1785
Epoch 040 -> Train SSE: 20.6250 | Val SSE: 7.0991
Epoch 050 -> Train SSE: 20.4219 | Val SSE: 7.0304
Epoch 060 -> Train SSE: 20.2450 | Val SSE: 6.9710
Epoch 070 -> Train SSE: 20.0912 | Val SSE: 6.9197
Epoch 080 -> Train SSE: 19.9579 | Val SSE: 6.8756
Epoch 090 -> Train SSE: 19.8427 | Val SSE: 6.8376
Epoch 100 -> Train SSE: 19.7438 | Val SSE: 6.8051
Epoch 110 -> Train SSE: 19.6590 | Val SSE: 6.7773
Epoch 120 -> Train SSE: 19.5865 | Val SSE: 6.7535
Epoch 130 -> Train SSE: 19.5245 | Val SSE: 6.7332
Epoch 140 -> Train SSE: 19.4716 | Val SSE: 6.7158
Epoch 150 -> Train SSE: 19.4262 | Val SSE: 6.7009
Epoch 160 -> Train SSE: 19.3872 | Val SSE: 6.6879
Epoch 170 -> Train SSE: 19.3533 | Val SSE: 6.6767
Epoch 180 -> Train SSE: 19.3238 | Val SSE: 6.6669
Epoch 190 -> Train SSE: 19.2977 | Val SSE: 6.6581
Epoch 200 -> Train SSE: 19.2744 | Val SSE: 6.6502


## h. Model Evaluation

In [87]:
# 1. Predict the probabilities for the test dataset
test_preds_proba = model.predict_proba(X_test_scaled)

# 2. Convert probabilities and one-hot true targets back to single class labels (0, 1, or 2)
# np.argmax gets the index of the highest probability/value per row
y_test_true_labels = np.argmax(y_test_encoded, axis=1)
y_test_pred_labels = np.argmax(test_preds_proba, axis=1)

# 3. Calculate the final accuracy score
test_accuracy = accuracy_score(y_test_true_labels, y_test_pred_labels)

# 4. Print the final results
print("=========================================")
print(f"Final Test Set Accuracy: {test_accuracy * 100:.2f}%")
print("=========================================")

Final Test Set Accuracy: 93.33%
